In [1]:
spark.sql("SHOW TABLES IN demo.bronze").show()

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|   bronze|   learner_profiles|      false|
|   bronze|    learning_events|      false|
|   bronze|  learning_feedback|      false|
|   bronze|      question_bank|      false|
|   bronze|reference_materials|      false|
+---------+-------------------+-----------+



In [2]:
spark.sql("""
SELECT 'learner_profiles' AS table_name, COUNT(*) AS row_count
FROM demo.bronze.learner_profiles

UNION ALL

SELECT 'question_bank', COUNT(*)
FROM demo.bronze.question_bank

UNION ALL

SELECT 'reference_materials', COUNT(*)
FROM demo.bronze.reference_materials

UNION ALL

SELECT 'learning_feedback', COUNT(*)
FROM demo.bronze.learning_feedback

UNION ALL

SELECT 'learning_events', COUNT(*)
FROM demo.bronze.learning_events
""").show()

+-------------------+---------+
|         table_name|row_count|
+-------------------+---------+
|   learner_profiles|        4|
|      question_bank|        5|
|reference_materials|        6|
|  learning_feedback|        9|
|    learning_events|        6|
+-------------------+---------+



In [3]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE WHEN user_id IS NULL
        THEN 1 ELSE 0 END
    ) AS null_user_id,

    SUM(
        CASE WHEN profile_updated_at IS NULL
        THEN 1 ELSE 0 END
    ) AS null_profile_updated_at,

    SUM(
        CASE WHEN ingestion_time IS NULL
        THEN 1 ELSE 0 END
    ) AS null_ingestion_time,

    SUM(
        CASE WHEN source_system IS NULL
        THEN 1 ELSE 0 END
    ) AS null_source_system,

    SUM(
        CASE
            WHEN raw_payload IS NULL
              OR TRIM(raw_payload) = ''
            THEN 1
            ELSE 0
        END
    ) AS empty_raw_payload

FROM demo.bronze.learner_profiles
""").show()

+----------+------------+-----------------------+-------------------+------------------+-----------------+
|total_rows|null_user_id|null_profile_updated_at|null_ingestion_time|null_source_system|empty_raw_payload|
+----------+------------+-----------------------+-------------------+------------------+-----------------+
|         4|           0|                      0|                  0|                 0|                0|
+----------+------------+-----------------------+-------------------+------------------+-----------------+



In [4]:
spark.sql("""
SELECT
    user_id,
    profile_updated_at,
    COUNT(*) AS duplicate_count
FROM demo.bronze.learner_profiles
GROUP BY
    user_id,
    profile_updated_at
HAVING COUNT(*) > 1
""").show()

+-------+------------------+---------------+
|user_id|profile_updated_at|duplicate_count|
+-------+------------------+---------------+
+-------+------------------+---------------+



In [5]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN from_json(
                raw_payload,
                'declared_background_level STRING,
                 learning_goal STRING,
                 main_domain STRING,
                 preferred_language STRING,
                 registration_date STRING'
            ) IS NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_json_rows,

    SUM(
        CASE
            WHEN ingestion_time < profile_updated_at
            THEN 1
            ELSE 0
        END
    ) AS invalid_time_order

FROM demo.bronze.learner_profiles
""").show()

+----------+-----------------+------------------+
|total_rows|invalid_json_rows|invalid_time_order|
+----------+-----------------+------------------+
|         4|                0|                 0|
+----------+-----------------+------------------+



In [6]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN question_id IS NULL THEN 1 ELSE 0 END)
        AS null_question_id,

    SUM(CASE WHEN question_version IS NULL THEN 1 ELSE 0 END)
        AS null_question_version,

    SUM(CASE WHEN source_system IS NULL THEN 1 ELSE 0 END)
        AS null_source_system,

    SUM(CASE WHEN created_at IS NULL THEN 1 ELSE 0 END)
        AS null_created_at,

    SUM(CASE WHEN ingestion_time IS NULL THEN 1 ELSE 0 END)
        AS null_ingestion_time,

    SUM(
        CASE
            WHEN raw_payload IS NULL
              OR TRIM(raw_payload) = ''
            THEN 1
            ELSE 0
        END
    ) AS empty_raw_payload

FROM demo.bronze.question_bank
""").show()

+----------+----------------+---------------------+------------------+---------------+-------------------+-----------------+
|total_rows|null_question_id|null_question_version|null_source_system|null_created_at|null_ingestion_time|empty_raw_payload|
+----------+----------------+---------------------+------------------+---------------+-------------------+-----------------+
|         5|               0|                    0|                 0|              0|                  0|                0|
+----------+----------------+---------------------+------------------+---------------+-------------------+-----------------+



In [7]:
spark.sql("""
SELECT
    question_id,
    question_version,
    COUNT(*) AS duplicate_count
FROM demo.bronze.question_bank
GROUP BY
    question_id,
    question_version
HAVING COUNT(*) > 1
""").show()

+-----------+----------------+---------------+
|question_id|question_version|duplicate_count|
+-----------+----------------+---------------+
+-----------+----------------+---------------+



In [8]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN from_json(
                raw_payload,
                'content_hash STRING,
                 correct_option_letter STRING,
                 created_by STRING,
                 difficulty_level INT,
                 domain STRING,
                 generation_model STRING,
                 is_active BOOLEAN,
                 option_a_text STRING,
                 option_b_text STRING,
                 option_c_text STRING,
                 option_d_text STRING,
                 question_text STRING,
                 question_type STRING,
                 subtopic STRING,
                 topic STRING,
                 validation_status STRING'
            ) IS NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_json_rows,

    SUM(
        CASE
            WHEN question_version <= 0
            THEN 1
            ELSE 0
        END
    ) AS invalid_question_version,

    SUM(
        CASE
            WHEN ingestion_time < created_at
            THEN 1
            ELSE 0
        END
    ) AS invalid_time_order

FROM demo.bronze.question_bank
""").show()

+----------+-----------------+------------------------+------------------+
|total_rows|invalid_json_rows|invalid_question_version|invalid_time_order|
+----------+-----------------+------------------------+------------------+
|         5|                0|                       0|                 0|
+----------+-----------------+------------------------+------------------+



In [9]:
spark.sql("""
WITH parsed AS (
    SELECT
        question_id,
        from_json(
            raw_payload,
            'content_hash STRING,
             correct_option_letter STRING,
             created_by STRING,
             difficulty_level INT,
             domain STRING,
             generation_model STRING,
             is_active BOOLEAN,
             option_a_text STRING,
             option_b_text STRING,
             option_c_text STRING,
             option_d_text STRING,
             question_text STRING,
             question_type STRING,
             subtopic STRING,
             topic STRING,
             validation_status STRING'
        ) AS payload
    FROM demo.bronze.question_bank
)

SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN payload.correct_option_letter NOT IN ('A', 'B', 'C', 'D')
              OR payload.correct_option_letter IS NULL
            THEN 1 ELSE 0
        END
    ) AS invalid_correct_option,

    SUM(
        CASE
            WHEN payload.question_text IS NULL
              OR TRIM(payload.question_text) = ''
            THEN 1 ELSE 0
        END
    ) AS empty_question_text,

    SUM(
        CASE
            WHEN payload.option_a_text IS NULL
              OR payload.option_b_text IS NULL
              OR payload.option_c_text IS NULL
              OR payload.option_d_text IS NULL
            THEN 1 ELSE 0
        END
    ) AS missing_options,

    SUM(
        CASE
            WHEN payload.difficulty_level IS NULL
              OR payload.difficulty_level NOT BETWEEN 1 AND 5
            THEN 1 ELSE 0
        END
    ) AS invalid_difficulty

FROM parsed
""").show()

+----------+----------------------+-------------------+---------------+------------------+
|total_rows|invalid_correct_option|empty_question_text|missing_options|invalid_difficulty|
+----------+----------------------+-------------------+---------------+------------------+
|         5|                     0|                  0|              0|                 0|
+----------+----------------------+-------------------+---------------+------------------+



In [10]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN reference_id IS NULL THEN 1 ELSE 0 END)
        AS null_reference_id,

    SUM(CASE WHEN batch_id IS NULL THEN 1 ELSE 0 END)
        AS null_batch_id,

    SUM(CASE WHEN source_type IS NULL THEN 1 ELSE 0 END)
        AS null_source_type,

    SUM(CASE WHEN source_name IS NULL THEN 1 ELSE 0 END)
        AS null_source_name,

    SUM(CASE WHEN file_name IS NULL THEN 1 ELSE 0 END)
        AS null_file_name,

    SUM(CASE WHEN import_time IS NULL THEN 1 ELSE 0 END)
        AS null_import_time,

    SUM(CASE WHEN ingestion_time IS NULL THEN 1 ELSE 0 END)
        AS null_ingestion_time,

    SUM(
        CASE
            WHEN raw_payload IS NULL
              OR TRIM(raw_payload) = ''
            THEN 1
            ELSE 0
        END
    ) AS empty_raw_payload

FROM demo.bronze.reference_materials
""").show()

+----------+-----------------+-------------+----------------+----------------+--------------+----------------+-------------------+-----------------+
|total_rows|null_reference_id|null_batch_id|null_source_type|null_source_name|null_file_name|null_import_time|null_ingestion_time|empty_raw_payload|
+----------+-----------------+-------------+----------------+----------------+--------------+----------------+-------------------+-----------------+
|         6|                0|            0|               0|               0|             0|               0|                  0|                0|
+----------+-----------------+-------------+----------------+----------------+--------------+----------------+-------------------+-----------------+



In [11]:
spark.sql("""
SELECT
    reference_id,
    COUNT(*) AS duplicate_count
FROM demo.bronze.reference_materials
GROUP BY reference_id
HAVING COUNT(*) > 1
""").show()

+------------+---------------+
|reference_id|duplicate_count|
+------------+---------------+
+------------+---------------+



In [12]:
spark.sql("""
WITH parsed AS (
    SELECT
        reference_id,
        import_time,
        ingestion_time,
        from_json(
            raw_payload,
            'author_or_owner STRING,
             content_text STRING,
             domain STRING,
             page_number INT,
             reliability_level STRING,
             section_name STRING,
             subtopic STRING,
             title STRING,
             topic STRING'
        ) AS payload
    FROM demo.bronze.reference_materials
)

SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN payload IS NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_json_rows,

    SUM(
        CASE
            WHEN ingestion_time < import_time
            THEN 1
            ELSE 0
        END
    ) AS invalid_time_order,

    SUM(
        CASE
            WHEN payload.content_text IS NULL
              OR TRIM(payload.content_text) = ''
            THEN 1
            ELSE 0
        END
    ) AS empty_content_text,

    SUM(
        CASE
            WHEN payload.title IS NULL
              OR TRIM(payload.title) = ''
            THEN 1
            ELSE 0
        END
    ) AS empty_title

FROM parsed
""").show()

+----------+-----------------+------------------+------------------+-----------+
|total_rows|invalid_json_rows|invalid_time_order|empty_content_text|empty_title|
+----------+-----------------+------------------+------------------+-----------+
|         6|                0|                 0|                 0|          0|
+----------+-----------------+------------------+------------------+-----------+



In [13]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN feedback_id IS NULL THEN 1 ELSE 0 END)
        AS null_feedback_id,

    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END)
        AS null_user_id,

    SUM(CASE WHEN session_id IS NULL THEN 1 ELSE 0 END)
        AS null_session_id,

    SUM(CASE WHEN feedback_stage IS NULL THEN 1 ELSE 0 END)
        AS null_feedback_stage,

    SUM(CASE WHEN feedback_time IS NULL THEN 1 ELSE 0 END)
        AS null_feedback_time,

    SUM(CASE WHEN ingestion_time IS NULL THEN 1 ELSE 0 END)
        AS null_ingestion_time,

    SUM(
        CASE
            WHEN raw_payload IS NULL
              OR TRIM(raw_payload) = ''
            THEN 1
            ELSE 0
        END
    ) AS empty_raw_payload

FROM demo.bronze.learning_feedback
""").show()

+----------+----------------+------------+---------------+-------------------+------------------+-------------------+-----------------+
|total_rows|null_feedback_id|null_user_id|null_session_id|null_feedback_stage|null_feedback_time|null_ingestion_time|empty_raw_payload|
+----------+----------------+------------+---------------+-------------------+------------------+-------------------+-----------------+
|         9|               0|           0|              0|                  0|                 0|                  0|                0|
+----------+----------------+------------+---------------+-------------------+------------------+-------------------+-----------------+



In [14]:
spark.sql("""
SELECT
    feedback_id,
    COUNT(*) AS duplicate_count
FROM demo.bronze.learning_feedback
GROUP BY feedback_id
HAVING COUNT(*) > 1
""").show()

+-----------+---------------+
|feedback_id|duplicate_count|
+-----------+---------------+
+-----------+---------------+



In [15]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN from_json(
                raw_payload,
                'confidence_after_score INT,
                 confidence_before_score INT,
                 expected_difficulty_score INT,
                 free_text STRING,
                 free_text_after STRING,
                 free_text_before STRING,
                 overall_confidence_score INT,
                 overall_motivation_score INT,
                 overall_stress_score INT,
                 perceived_difficulty_score INT,
                 perceived_understanding_after_score INT,
                 perceived_understanding_before_score INT,
                 still_confused BOOLEAN,
                 topics_feedback ARRAY<STRUCT<
                     confidence_score: INT,
                     perceived_understanding_score: INT,
                     still_confused: BOOLEAN,
                     topic_id: STRING
                 >>'
            ) IS NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_json_rows,

    SUM(
        CASE
            WHEN ingestion_time < feedback_time
            THEN 1
            ELSE 0
        END
    ) AS invalid_time_order,

    SUM(
        CASE
            WHEN feedback_stage NOT IN (
                'before_practice',
                'after_practice',
                'general_check_in'
            )
            THEN 1
            ELSE 0
        END
    ) AS invalid_feedback_stage,

    SUM(
        CASE
            WHEN feedback_stage IN (
                'before_practice',
                'after_practice'
            )
            AND practice_id IS NULL
            THEN 1
            ELSE 0
        END
    ) AS missing_practice_id,

    SUM(
        CASE
            WHEN feedback_stage = 'general_check_in'
            AND practice_id IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS unexpected_practice_id

FROM demo.bronze.learning_feedback
""").show()

+----------+-----------------+------------------+----------------------+-------------------+----------------------+
|total_rows|invalid_json_rows|invalid_time_order|invalid_feedback_stage|missing_practice_id|unexpected_practice_id|
+----------+-----------------+------------------+----------------------+-------------------+----------------------+
|         9|                0|                 0|                     0|                  0|                     0|
+----------+-----------------+------------------+----------------------+-------------------+----------------------+



In [16]:
spark.sql("""
WITH parsed AS (
    SELECT
        feedback_stage,
        from_json(
            raw_payload,
            'confidence_after_score INT,
             confidence_before_score INT,
             expected_difficulty_score INT,
             free_text STRING,
             free_text_after STRING,
             free_text_before STRING,
             overall_confidence_score INT,
             overall_motivation_score INT,
             overall_stress_score INT,
             perceived_difficulty_score INT,
             perceived_understanding_after_score INT,
             perceived_understanding_before_score INT,
             still_confused BOOLEAN,
             topics_feedback ARRAY<STRUCT<
                 confidence_score: INT,
                 perceived_understanding_score: INT,
                 still_confused: BOOLEAN,
                 topic_id: STRING
             >>'
        ) AS payload
    FROM demo.bronze.learning_feedback
)

SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN payload.confidence_before_score IS NOT NULL
             AND payload.confidence_before_score NOT BETWEEN 1 AND 10
            THEN 1 ELSE 0
        END
    ) AS invalid_confidence_before,

    SUM(
        CASE
            WHEN payload.confidence_after_score IS NOT NULL
             AND payload.confidence_after_score NOT BETWEEN 1 AND 10
            THEN 1 ELSE 0
        END
    ) AS invalid_confidence_after,

    SUM(
        CASE
            WHEN payload.expected_difficulty_score IS NOT NULL
             AND payload.expected_difficulty_score NOT BETWEEN 1 AND 10
            THEN 1 ELSE 0
        END
    ) AS invalid_expected_difficulty,

    SUM(
        CASE
            WHEN payload.perceived_difficulty_score IS NOT NULL
             AND payload.perceived_difficulty_score NOT BETWEEN 1 AND 10
            THEN 1 ELSE 0
        END
    ) AS invalid_perceived_difficulty,

    SUM(
        CASE
            WHEN payload.overall_confidence_score IS NOT NULL
             AND payload.overall_confidence_score NOT BETWEEN 1 AND 10
            THEN 1 ELSE 0
        END
    ) AS invalid_overall_confidence,

    SUM(
        CASE
            WHEN payload.overall_motivation_score IS NOT NULL
             AND payload.overall_motivation_score NOT BETWEEN 1 AND 10
            THEN 1 ELSE 0
        END
    ) AS invalid_overall_motivation,

    SUM(
        CASE
            WHEN payload.overall_stress_score IS NOT NULL
             AND payload.overall_stress_score NOT BETWEEN 1 AND 10
            THEN 1 ELSE 0
        END
    ) AS invalid_overall_stress

FROM parsed
""").show()

+----------+-------------------------+------------------------+---------------------------+----------------------------+--------------------------+--------------------------+----------------------+
|total_rows|invalid_confidence_before|invalid_confidence_after|invalid_expected_difficulty|invalid_perceived_difficulty|invalid_overall_confidence|invalid_overall_motivation|invalid_overall_stress|
+----------+-------------------------+------------------------+---------------------------+----------------------------+--------------------------+--------------------------+----------------------+
|         9|                        0|                       0|                          0|                           0|                         0|                         0|                     0|
+----------+-------------------------+------------------------+---------------------------+----------------------------+--------------------------+--------------------------+----------------------+



In [17]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN event_id IS NULL THEN 1 ELSE 0 END)
        AS null_event_id,

    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END)
        AS null_user_id,

    SUM(CASE WHEN session_id IS NULL THEN 1 ELSE 0 END)
        AS null_session_id,

    SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END)
        AS null_event_type,

    SUM(CASE WHEN event_time IS NULL THEN 1 ELSE 0 END)
        AS null_event_time,

    SUM(CASE WHEN ingestion_time IS NULL THEN 1 ELSE 0 END)
        AS null_ingestion_time,

    SUM(CASE WHEN source_system IS NULL THEN 1 ELSE 0 END)
        AS null_source_system,

    SUM(
        CASE
            WHEN raw_payload IS NULL
              OR TRIM(raw_payload) = ''
            THEN 1
            ELSE 0
        END
    ) AS empty_raw_payload

FROM demo.bronze.learning_events
""").show()

+----------+-------------+------------+---------------+---------------+---------------+-------------------+------------------+-----------------+
|total_rows|null_event_id|null_user_id|null_session_id|null_event_type|null_event_time|null_ingestion_time|null_source_system|empty_raw_payload|
+----------+-------------+------------+---------------+---------------+---------------+-------------------+------------------+-----------------+
|         6|            0|           0|              0|              0|              0|                  0|                 0|                0|
+----------+-------------+------------+---------------+---------------+---------------+-------------------+------------------+-----------------+



In [18]:
spark.sql("""
SELECT
    event_id,
    COUNT(*) AS duplicate_count
FROM demo.bronze.learning_events
GROUP BY event_id
HAVING COUNT(*) > 1
""").show()

+--------+---------------+
|event_id|duplicate_count|
+--------+---------------+
+--------+---------------+



In [19]:
spark.sql("""
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN from_json(
                raw_payload,
                'answers ARRAY<STRUCT<
                    attempt_duration_seconds: INT,
                    hints_used: INT,
                    question_id: STRING,
                    question_version: INT,
                    selected_option_letter: STRING
                 >>,
                 assistant_answer STRING,
                 completion_status STRING,
                 conversation_id STRING,
                 conversation_summary STRING,
                 detected_concepts ARRAY<STRING>,
                 difficulty_indicators ARRAY<STRING>,
                 important_points ARRAY<STRING>,
                 learner_intent STRING,
                 possible_confusion BOOLEAN,
                 practice_id STRING,
                 processing_model STRING,
                 started_at STRING,
                 submitted_at STRING,
                 topic_id STRING,
                 user_prompt STRING'
            ) IS NULL
            THEN 1
            ELSE 0
        END
    ) AS invalid_json_rows,

    SUM(
        CASE
            WHEN ingestion_time < event_time
            THEN 1
            ELSE 0
        END
    ) AS invalid_time_order,

    SUM(
        CASE
            WHEN event_type NOT IN (
                'ai_learning_interaction',
                'practice_submitted'
            )
            THEN 1
            ELSE 0
        END
    ) AS invalid_event_type,

    SUM(
        CASE
            WHEN event_type = 'ai_learning_interaction'
             AND source_system <> 'chat'
            THEN 1
            WHEN event_type = 'practice_submitted'
             AND source_system <> 'practice_app'
            THEN 1
            ELSE 0
        END
    ) AS invalid_source_for_event

FROM demo.bronze.learning_events
""").show()

+----------+-----------------+------------------+------------------+------------------------+
|total_rows|invalid_json_rows|invalid_time_order|invalid_event_type|invalid_source_for_event|
+----------+-----------------+------------------+------------------+------------------------+
|         6|                0|                 0|                 0|                       0|
+----------+-----------------+------------------+------------------+------------------------+

